# Load postcodes into the `postcode` table

Reads the `SO` file from the ONS Postcode Directory (ONSPD) and fills the `postcode` table. Each postcode is linked to one row in `area` through its 2021 LSOA code. The Business and Venues loaders will use this table to place a company or unit in an area.

Run `populate_area.ipynb` first, because this notebook only keeps postcodes whose LSOA is in the `area` table.

Notes
- **Terminated postcodes are kept.** About 60% of postcodes in the file have been retired. Older companies still list them, so dropping them would leave those companies with no area. `terminated` is empty for postcodes still in use.
- Postcode text is stored uppercase with a single space (`SO14 2AB`). Normalise postcodes from any other source the same way before matching.
- `introduced` and `terminated` are months stored as text (`1997-06`).
- If you later add fringe councils to `area` (Eastleigh, Totton), re-run this notebook and their postcodes are added. Existing `id` values never change.
- Only the `SO` file is loaded. A Southampton LSOA that contains a postcode from another postcode area would be missed, but that should be very rare.
- Safe to re-run: rows are matched on `postcode`.

## 1. Settings

In [1]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

CSV_PATH = Path("raw/official/ONSPD_MAY_2026_UK_SO.csv")   # the SO file from the ONSPD zip
DB_PATH = Path("processed/database.db")       # same database as the other notebooks

## 2. Read the file

In [2]:
needed = ["pcds", "dointr", "doterm", "lat", "long", "lsoa21cd"]

# Check the columns first, so a changed file gives a clear message
header = pd.read_csv(CSV_PATH, nrows=0, encoding="utf-8-sig").columns.str.strip().tolist()
missing = [c for c in needed if c not in header]
assert not missing, f"Columns missing from the file: {missing}"

raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig", usecols=needed)
print(f"{len(raw):,} postcodes read")
raw.head(3)

40,875 postcodes read


,pcds,dointr,doterm,lat,long,lsoa21cd
0,SO1 0AA,198001,199412,50.899063,-1.404065,E01032748
1,SO1 0AB,198001,199412,50.899398,-1.404587,E01032748
2,SO1 0AD,198001,199412,50.899807,-1.405407,E01032748


## 3. Link each postcode to an area

In [3]:
con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON;")   # SQLite ignores foreign keys unless this is set on every connection
con.execute("PRAGMA journal_mode = WAL;")  # lets teammates read while you write

tables = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table'")}
assert "area" in tables, "There is no area table yet. Run populate_area.ipynb first."
area_ids = dict(con.execute("SELECT lsoa_code, id FROM area"))
assert area_ids, "The area table is empty. Run populate_area.ipynb first."

pc = raw.copy()
pc["postcode"] = pc["pcds"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
pc["lsoa21cd"] = pc["lsoa21cd"].str.strip()
pc["area_id"] = pc["lsoa21cd"].map(area_ids)

blank_lsoa = int((pc["lsoa21cd"] == "").sum())
outside = int(pc["area_id"].isna().sum()) - blank_lsoa
pc = pc[pc["area_id"].notna()].copy()

print(f"{len(raw):,} postcodes in the file")
print(f"{len(pc):,} kept (their LSOA is in the area table)")
print(f"{outside:,} dropped: in an area you haven't loaded (other councils)")
print(f"{blank_lsoa:,} dropped: no LSOA recorded")

40,875 postcodes in the file
14,384 kept (their LSOA is in the area table)
26,380 dropped: in an area you haven't loaded (other councils)
111 dropped: no LSOA recorded


## 4. Validate

In [4]:
bad_format = pc[~pc["postcode"].str.fullmatch(r"[A-Z]{1,2}\d[A-Z\d]? \d[A-Z]{2}")]
assert bad_format.empty, f"Postcodes in an unexpected format:\n{bad_format['postcode'].head()}"
assert pc["postcode"].is_unique, "Duplicate postcodes in the selection"

pc["lat"] = pd.to_numeric(pc["lat"], errors="coerce")
pc["lon"] = pd.to_numeric(pc["long"], errors="coerce")
# ONSPD uses 99.999999 when a postcode has no coordinates; this range also catches blanks
bad_coords = pc[~(pc["lat"].between(49.5, 61) & pc["lon"].between(-9, 2.5))]
assert bad_coords.empty, f"Postcodes with missing or impossible coordinates:\n{bad_coords[['postcode', 'lat', 'lon']].head()}"

for col in ["dointr", "doterm"]:
    ok = pc[col].str.fullmatch(r"\d{6}") | (pc[col] == "")
    assert ok.all(), f"{col} has values that aren't YYYYMM:\n{pc.loc[~ok, col].head()}"
assert (pc["dointr"] != "").all(), "Some postcodes have no introduced date"

bad_dates = pc[(pc["doterm"] != "") & (pc["doterm"] < pc["dointr"])]
assert bad_dates.empty, f"Postcodes terminated before they were introduced:\n{bad_dates[['postcode', 'dointr', 'doterm']].head()}"

live = int((pc["doterm"] == "").sum())
print(f"{len(pc):,} postcodes pass the checks ({live:,} in use, {len(pc) - live:,} terminated)")

14,384 postcodes pass the checks (5,806 in use, 8,578 terminated)


## 5. Create the table

In [5]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS postcode (
    id         INTEGER PRIMARY KEY,
    area_id    INTEGER NOT NULL REFERENCES area(id),
    postcode   TEXT NOT NULL UNIQUE,   -- uppercase, one space: 'SO14 2AB'
    lat        REAL NOT NULL,
    lon        REAL NOT NULL,
    introduced TEXT,                   -- 'YYYY-MM'
    terminated TEXT                    -- 'YYYY-MM', empty (NULL) if still in use
);
CREATE INDEX IF NOT EXISTS idx_postcode_area ON postcode(area_id);
"""
con.executescript(SCHEMA)

## 6. Load the rows

In [6]:
def month(v):
    # '199706' -> '1997-06', blank -> NULL
    return f"{v[:4]}-{v[4:6]}" if v else None


# Plain Python types, because SQLite can't take numpy numbers
rows = [
    (int(r.area_id), r.postcode, float(r.lat), float(r.lon), month(r.dointr), month(r.doterm))
    for r in pc.itertuples(index=False)
]

with con:   # one transaction: all rows are saved, or none are
    con.executemany(
        """
        INSERT INTO postcode (area_id, postcode, lat, lon, introduced, terminated)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT(postcode) DO UPDATE SET
            area_id    = excluded.area_id,
            lat        = excluded.lat,
            lon        = excluded.lon,
            introduced = excluded.introduced,
            terminated = excluded.terminated
        """,
        rows,
    )

print(f"Loaded {len(rows):,} rows into {DB_PATH}")

Loaded 14,384 rows into processed/database.db


## 7. Check the result

In [7]:
total, live = con.execute("SELECT COUNT(*), SUM(terminated IS NULL) FROM postcode").fetchone()
print(f"{total:,} postcodes in the table ({live:,} in use, {total - live:,} terminated)")

violations = con.execute("PRAGMA foreign_key_check").fetchall()
assert not violations, f"Broken foreign keys: {violations[:5]}"

no_postcodes = con.execute(
    """
    SELECT COUNT(*) FROM area a
    WHERE NOT EXISTS (SELECT 1 FROM postcode p WHERE p.area_id = a.id)
    """
).fetchone()[0]
print(f"Areas with no postcodes at all: {no_postcodes}")

14,384 postcodes in the table (5,806 in use, 8,578 terminated)
Areas with no postcodes at all: 0


Postcodes in use, per ward. Every ward should have some:

In [8]:
pd.read_sql(
    """
    SELECT a.ward_name,
           COUNT(*)                       AS postcodes,
           SUM(p.terminated IS NULL)      AS in_use
    FROM postcode p
    JOIN area a ON a.id = p.area_id
    GROUP BY a.ward_name
    ORDER BY a.ward_name
    """,
    con,
)

,ward_name,postcodes,in_use
0,Banister & Polygon,1229,439
1,Bargate,1401,483
2,Bassett,790,388
3,Bevois,962,384
4,Bitterne Park,1472,429
5,Coxford,480,232
6,Freemantle,1384,326
7,Harefield,652,318
8,Millbrook,786,361
9,Peartree,621,282


This is the join the Business loader will use to find a company's area from its postcode:

In [9]:
pd.read_sql(
    """
    SELECT p.postcode, a.name AS area, a.ward_name
    FROM postcode p
    JOIN area a ON a.id = p.area_id
    WHERE p.terminated IS NULL
    LIMIT 5
    """,
    con,
)

,postcode,area,ward_name
0,SO14 0AA,Southampton 033B,Bargate
1,SO14 0AB,Southampton 033B,Bargate
2,SO14 0AE,Southampton 022A,Bevois
3,SO14 0AF,Southampton 022B,Bevois
4,SO14 0AG,Southampton 022G,Bevois


In [10]:
con.close()